# Sarvam-30B → GGUF

Converts `sarvamai/sarvam-30b` to GGUF by applying [llama.cpp PR #20275](https://github.com/ggml-org/llama.cpp/pull/20275) which adds `sarvam_moe` architecture support.

**Instructions:** Runtime → Change runtime type → GPU → **Run All**

Model stores on Google Drive (survives disconnects). GGUF output also goes to Drive.

## Step 1: Mount Drive + Install deps

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
MODEL_DIR = "/content/drive/MyDrive/sarvam-gguf/sarvam-30b"
OUTPUT_DIR = "/content/drive/MyDrive/sarvam-gguf/output"
LLAMA_CPP_DIR = "/content/llama.cpp"
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

!pip install -q huggingface_hub hf_transfer sentencepiece protobuf transformers numpy torch gguf

_, _, free = shutil.disk_usage("/content")
_, _, free_d = shutil.disk_usage("/content/drive/MyDrive")
print(f"Local: {free/(1024**3):.0f}GB | Drive: {free_d/(1024**3):.0f}GB")
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || echo 'CPU only'

## Step 2: Clone llama.cpp + apply PR #20275

In [ ]:
import subprocess, os
LLAMA_CPP_DIR = "/content/llama.cpp"

if not os.path.exists(LLAMA_CPP_DIR):
    !git clone https://github.com/ggml-org/llama.cpp {LLAMA_CPP_DIR}

os.chdir(LLAMA_CPP_DIR)

# Check if PR already applied
r = subprocess.run(["grep", "-q", "SarvamMoEForCausalLM", "convert_hf_to_gguf.py"])
if r.returncode == 0:
    print("✅ PR #20275 already applied")
else:
    print("Applying PR #20275 (sarvam_moe architecture)...")
    # Fetch the PR branch and cherry-pick
    !git remote add pr20275 https://github.com/sumitchatterjee13/llama.cpp.git 2>/dev/null; \
     git fetch pr20275 sarvam_moe_arch_support --depth=20 2>/dev/null
    # Get the PR commits
    !curl -sL https://github.com/ggml-org/llama.cpp/pull/20275.patch > /tmp/sarvam.patch
    !git apply --check /tmp/sarvam.patch 2>&1 && git apply /tmp/sarvam.patch && echo '✅ Patch applied' || echo '❌ Patch failed — may need manual merge'

# Verify
!grep -c 'SarvamMoEForCausalLM' convert_hf_to_gguf.py && echo '✅ SarvamMoEForCausalLM registered' || echo '❌ NOT registered'

os.chdir('/content')

## Step 3: Build llama.cpp

In [ ]:
%%bash
cd /content/llama.cpp

# Build with CUDA if available, else CPU
if command -v nvcc &>/dev/null; then
    CUDA_FLAG="-DGGML_CUDA=ON"
    echo "Building with CUDA..."
else
    CUDA_FLAG="-DGGML_CUDA=OFF"
    echo "Building CPU-only..."
fi

cmake -B build -DBUILD_SHARED_LIBS=OFF $CUDA_FLAG -DCMAKE_BUILD_TYPE=Release 2>&1 | tail -5
cmake --build build --config Release -j$(nproc) --target llama-quantize llama-cli 2>&1 | tail -5

# Verify
for t in llama-quantize llama-cli; do
    [ -f "build/bin/$t" ] && echo "✅ $t" || echo "❌ $t missing"
done

## Step 4: Download Sarvam-30B

In [ ]:
import os
MODEL_DIR = "/content/drive/MyDrive/sarvam-gguf/sarvam-30b"

if os.path.exists(f"{MODEL_DIR}/config.json"):
    print("✅ Already downloaded")
else:
    print("Downloading sarvamai/sarvam-30b to Google Drive (~60GB)...")
    print("If interrupted, re-run this cell to resume.\n")
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id="sarvamai/sarvam-30b",
        local_dir=MODEL_DIR,
        ignore_patterns=["*.bin", "*.h5", "*.msgpack", "original/**"],
        resume_download=True,
    )
    !rm -rf /content/hf_cache 2>/dev/null
    print("✅ Done")

!du -sh {MODEL_DIR}
!ls {MODEL_DIR}/*.safetensors 2>/dev/null | wc -l | xargs -I {} echo '{} safetensor shards'

## Step 5: Convert to GGUF (F16)

In [ ]:
import os, time
MODEL_DIR = "/content/drive/MyDrive/sarvam-gguf/sarvam-30b"
OUTPUT_DIR = "/content/drive/MyDrive/sarvam-gguf/output"
LLAMA_CPP_DIR = "/content/llama.cpp"
GGUF_F16 = f"{OUTPUT_DIR}/sarvam-30b-f16.gguf"

if os.path.exists(GGUF_F16):
    print(f"✅ F16 GGUF already exists ({os.path.getsize(GGUF_F16)/(1024**3):.1f} GB)")
else:
    print("Converting Sarvam-30B → GGUF F16...\n")
    start = time.time()
    !python3 {LLAMA_CPP_DIR}/convert_hf_to_gguf.py \
        {MODEL_DIR} \
        --outfile {GGUF_F16} \
        --outtype f16
    elapsed = time.time() - start
    if os.path.exists(GGUF_F16):
        print(f"\n✅ SUCCESS: {os.path.getsize(GGUF_F16)/(1024**3):.1f} GB in {elapsed/60:.0f} min")
    else:
        print(f"\n❌ FAILED after {elapsed/60:.0f} min")

## Step 6: Quantize to Q4_K_M

In [ ]:
import os
OUTPUT_DIR = "/content/drive/MyDrive/sarvam-gguf/output"
GGUF_F16 = f"{OUTPUT_DIR}/sarvam-30b-f16.gguf"
QUANTIZE = "/content/llama.cpp/build/bin/llama-quantize"

if not os.path.exists(GGUF_F16):
    print("❌ No F16 GGUF — run Step 5 first")
else:
    for method in ["q4_k_m", "q8_0"]:
        out = f"{OUTPUT_DIR}/sarvam-30b-{method}.gguf"
        if os.path.exists(out):
            print(f"✅ {method} already exists ({os.path.getsize(out)/(1024**3):.1f} GB)")
            continue
        print(f"Quantizing to {method}...")
        !{QUANTIZE} {GGUF_F16} {out} {method}
        if os.path.exists(out):
            print(f"✅ {method}: {os.path.getsize(out)/(1024**3):.1f} GB\n")
        else:
            print(f"❌ {method} failed\n")

    print("\n📁 Output:")
    !ls -lh {OUTPUT_DIR}/*.gguf 2>/dev/null

## Step 7: Test it

In [ ]:
import os
OUTPUT_DIR = "/content/drive/MyDrive/sarvam-gguf/output"
GGUF = f"{OUTPUT_DIR}/sarvam-30b-q4_k_m.gguf"
CLI = "/content/llama.cpp/build/bin/llama-cli"

if not os.path.exists(GGUF):
    print("❌ No quantized GGUF to test")
else:
    print("Testing Sarvam-30B Q4_K_M...\n")
    !{CLI} \
        --model {GGUF} \
        --n-gpu-layers 99 \
        --ctx-size 2048 \
        --temp 0.7 \
        -no-cnv \
        --prompt "भारत के बारे में बताइए।"